In [44]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/azrinsultana/zfindata/zfin.txt
/kaggle/input/datasets/azrinsultana/zebrafish/uniprotkb_zebra_fish_2026_04_21.xlsx
/kaggle/input/datasets/azrinsultana/uniport-human/uniprotkb_Homo_sapiens_human_AND_model_2026_03_08.tsv


# Uniprot dataset

In [4]:
import pandas as pd

file_path = "/kaggle/input/datasets/azrinsultana/uniport-human/uniprotkb_Homo_sapiens_human_AND_model_2026_03_08.tsv"

df = pd.read_csv(file_path, sep="\t")

print(df.head())

        Entry  Reviewed   Entry Name  \
0  A0A087X1C5  reviewed  CP2D7_HUMAN   
1  A0A096LP01  reviewed  SIM26_HUMAN   
2  A0A0B4J2F0  reviewed  PIOS1_HUMAN   
3  A0A0C5B5G6  reviewed  MOTSC_HUMAN   
4  A0A0K2S4Q6  reviewed  CD3CH_HUMAN   

                                       Protein names        Gene Names  \
0                 Cytochrome P450 2D7 (EC 1.14.14.1)            CYP2D7   
1                 Small integral membrane protein 26  SMIM26 LINC00493   
2   Protein PIGBOS1 (PIGB opposite strand protein 1)           PIGBOS1   
3  Mitochondrial-derived peptide MOTS-c (Mitochon...           MT-RNR1   
4  Protein CD300H (CD300 antigen-like family memb...            CD300H   

               Organism  Length  \
0  Homo sapiens (Human)     515   
1  Homo sapiens (Human)      95   
2  Homo sapiens (Human)      54   
3  Homo sapiens (Human)      16   
4  Homo sapiens (Human)     201   

                                       Function [CC] Pathway UniPathway  \
0  FUNCTION: May be responsi

In [5]:
df.shape

(20431, 12)

In [6]:
df.isnull().sum()

Entry                            0
Reviewed                         0
Entry Name                       0
Protein names                    0
Gene Names                     125
Organism                         0
Length                           0
Function [CC]                 3172
Pathway                      19050
UniPathway                   19081
PathwayCommons                 998
Subcellular location [CC]     3126
dtype: int64

In [11]:
df = df[df["Function [CC]"].notnull()]
df = df.loc[:, df.isnull().mean() <= 0.55]

df = df.reset_index(drop=True)

print(df.shape)
print(df.isnull().sum())

(17259, 11)
Entry                           0
Reviewed                        0
Entry Name                      0
Protein names                   0
Gene Names                     14
Organism                        0
Length                          0
Function [CC]                   0
PathwayCommons                476
Subcellular location [CC]    1321
clean_function                  0
dtype: int64


In [9]:
df["Function [CC]"].iloc[3]

"FUNCTION: Regulates insulin sensitivity and metabolic homeostasis (PubMed:25738459, PubMed:33468709). Inhibits the folate cycle, thereby reducing de novo purine biosynthesis which leads to the accumulation of the de novo purine synthesis intermediate 5-aminoimidazole-4-carboxamide (AICAR) and the activation of the metabolic regulator 5'-AMP-activated protein kinase (AMPK) (PubMed:25738459). Protects against age-dependent and diet-induced insulin resistance as well as diet-induced obesity (PubMed:25738459). In response to metabolic stress, translocates to the nucleus where it binds to antioxidant response elements (ARE) present in the promoter regions of a number of genes and plays a role in regulating nuclear gene expression in an NFE2L2-dependent manner and increasing cellular resistance to metabolic stress (PubMed:29983246). Increases mitochondrial respiration and levels of CPT1A and cytokines IL1B, IL6, IL8, IL10 and TNF in senescent cells (PubMed:29886458). Increases activity of t

In [13]:
import re
import pandas as pd

def clean_function(text):
    if pd.isna(text):
        return text
    
    text = re.sub(r"FUNCTION:\s*", "", text)
        text = re.sub(r"\{.*?\}", "", text)
        text = re.sub(r"\s*\([^)]*\)\.?", "", text)
    
    return text.strip()

df["clean_function"] = df["Function [CC]"].apply(clean_function)

In [14]:
df["clean_function"].iloc[3]

"Regulates insulin sensitivity and metabolic homeostasis Inhibits the folate cycle, thereby reducing de novo purine biosynthesis which leads to the accumulation of the de novo purine synthesis intermediate 5-aminoimidazole-4-carboxamide and the activation of the metabolic regulator 5'-AMP-activated protein kinase Protects against age-dependent and diet-induced insulin resistance as well as diet-induced obesity In response to metabolic stress, translocates to the nucleus where it binds to antioxidant response elements present in the promoter regions of a number of genes and plays a role in regulating nuclear gene expression in an NFE2L2-dependent manner and increasing cellular resistance to metabolic stress Increases mitochondrial respiration and levels of CPT1A and cytokines IL1B, IL6, IL8, IL10 and TNF in senescent cells Increases activity of the serine/threonine protein kinase complex mTORC2 and reduces activity of the PTEN phosphatase, thus promoting phosphorylation of AKT This prom

In [17]:
import re
import pandas as pd

def extract_pubmed_ids(text):
    if pd.isna(text):
        return None
    
    ids = re.findall(r"PubMed:(\d+)", text)
    
    return ",".join(ids) if ids else None

df["pubmed_ids"] = df["Function [CC]"].apply(extract_pubmed_ids)

In [18]:
df.head()

,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Function [CC],PathwayCommons,Subcellular location [CC],clean_function,pubmed_ids
0,A0A087X1C5,reviewed,CP2D7_HUMAN,Cytochrome P450 2D7 (EC 1.14.14.1),CYP2D7,Homo sapiens (Human),515,FUNCTION: May be responsible for the metabolis...,A0A087X1C5;,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,May be responsible for the metabolism of many ...,"15051713,18838503,15051713,18838503"
1,A0A096LP01,reviewed,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,Homo sapiens (Human),95,FUNCTION: May play a role in cell viability. {...,NaN,SUBCELLULAR LOCATION: Mitochondrion outer memb...,May play a role in cell viability. .,34445188
2,A0A0B4J2F0,reviewed,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,FUNCTION: Plays a role in regulation of the un...,A0A0B4J2F0;,SUBCELLULAR LOCATION: Mitochondrion outer memb...,Plays a role in regulation of the unfolded pro...,31653868
3,A0A0C5B5G6,reviewed,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,FUNCTION: Regulates insulin sensitivity and me...,NaN,SUBCELLULAR LOCATION: Secreted {ECO:0000269|Pu...,Regulates insulin sensitivity and metabolic ho...,"25738459,33468709,25738459,25738459,29983246,2..."
4,A0A0K2S4Q6,reviewed,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,Homo sapiens (Human),201,FUNCTION: May play an important role in innate...,A0A0K2S4Q6;,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...,May play an important role in innate immunity ...,26221034


In [19]:
df.to_csv("/kaggle/working/cleaned_uniprot.tsv", sep="\t", index=False)

In [20]:
df.columns

Index(['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Function [CC]', 'PathwayCommons',
       'Subcellular location [CC]', 'clean_function', 'pubmed_ids'],
      dtype='object')

## collecting abstract

In [21]:
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.2 MB/s eta 0:00:00a 0:00:01


In [22]:
import re
import time
import pandas as pd
from Bio import Entrez

Entrez.email = "your_email@example.com" 




def extract_pubmed_ids(value):
    if pd.isna(value):
        return []

    ids = re.findall(r"\d+", str(value))
    return list(dict.fromkeys(ids))


def parse_article(article):
    medline = article["MedlineCitation"]
    article_data = medline["Article"]

    pmid = str(medline["PMID"])

    title = str(article_data.get("ArticleTitle", ""))

    journal = str(article_data.get("Journal", {}).get("Title", ""))

    abstract = ""
    abstract_parts = article_data.get("Abstract", {}).get("AbstractText", [])

    if abstract_parts:
        abstract = " ".join([str(part) for part in abstract_parts])

    year = ""
    try:
        pub_date = article_data["Journal"]["JournalIssue"]["PubDate"]
        year = str(pub_date.get("Year", pub_date.get("MedlineDate", "")))
    except Exception:
        pass

    # DOI
    doi = ""
    try:
        for article_id in article["PubmedData"]["ArticleIdList"]:
            if article_id.attributes.get("IdType") == "doi":
                doi = str(article_id)
                break
    except Exception:
        pass

    return {
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "journal": journal,
        "year": year,
        "doi": doi
    }


def fetch_pubmed_details(pubmed_ids, batch_size=100, sleep_time=0.4):
    pubmed_ids = sorted(set(pubmed_ids))
    all_articles = []

    for i in range(0, len(pubmed_ids), batch_size):
        batch = pubmed_ids[i:i + batch_size]

        try:
            handle = Entrez.efetch(
                db="pubmed",
                id=",".join(batch),
                retmode="xml"
            )

            records = Entrez.read(handle, validate=False)
            handle.close()

            for article in records["PubmedArticle"]:
                all_articles.append(parse_article(article))

        except Exception as e:
            print(f"Error fetching batch {i // batch_size + 1}: {e}")

        time.sleep(sleep_time)

    return pd.DataFrame(all_articles)



all_pubmed_ids = []

for value in df["pubmed_ids"]:
    all_pubmed_ids.extend(extract_pubmed_ids(value))

all_pubmed_ids = sorted(set(all_pubmed_ids))

print("Total unique PubMed IDs:", len(all_pubmed_ids))
pubmed_df = fetch_pubmed_details(all_pubmed_ids)

print("PubMed dataset shape:", pubmed_df.shape)

pubmed_df.to_csv("pubmed_details_dataset.csv", index=False)

pubmed_df.to_json(
    "pubmed_details_dataset.json",
    orient="records",
    indent=2,
    force_ascii=False
)

pubmed_df.head()

Total unique PubMed IDs: 32909
PubMed dataset shape: (32909, 6)


,pmid,title,abstract,journal,year,doi
0,10021369,"Identification of APC2, a homologue of the ade...",The adenomatous polyposis coli (APC) tumour-su...,Current biology : CB,1999,10.1016/s0960-9822(99)80024-4
1,10021370,Alpha-kinases: a new class of protein kinases ...,,Current biology : CB,1999,10.1016/s0960-9822(99)80006-2
2,10022120,Tyrosine phosphorylation and complex formation...,"Cbl-b, a mammalian homolog of Cbl, consists of...",Oncogene,1999,10.1038/sj.onc.1202411
3,10022127,"TIF1gamma, a novel member of the transcription...",We report the cloning and characterization of ...,Oncogene,1999,10.1038/sj.onc.1202655
4,10022843,"The splicing factor-associated protein, p32, r...",The cellular protein p32 was isolated original...,The EMBO journal,1999,10.1093/emboj/18.4.1014


In [23]:
pubmed_df['abstract']

0        The adenomatous polyposis coli (APC) tumour-su...
1                                                         
2        Cbl-b, a mammalian homolog of Cbl, consists of...
3        We report the cloning and characterization of ...
4        The cellular protein p32 was isolated original...
                               ...                        
32904    Based on amino acid sequence information from ...
32905    Microtubules are involved in the positioning a...
32906    Previously we cloned a novel adaptor protein, ...
32907    The existence of human renin-binding protein (...
32908    Ubiquitin-mediated proteolysis has a central r...
Name: abstract, Length: 32909, dtype: object

In [24]:
pubmed_df.to_csv("/kaggle/working/pubmed_abstarct.tsv", sep="\t", index=False)

# Zebra fish data

In [20]:
import pandas as pd

zfin_path = "/kaggle/input/datasets/azrinsultana/zfindata/zfin.txt"

zfin_df = pd.read_csv(
    zfin_path,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print(zfin_df.shape)
print(zfin_df.columns)
zfin_df.head()

(44010, 14)
Index(['ZFIN ID', 'ZFIN Symbol', 'ZFIN Name', 'Human Symbol', 'Human Name',
       'OMIM ID', 'Gene ID', 'HGNC ID', 'Evidence', 'Pub ID',
       'ZFIN Abbreviation Name', 'ECO ID', 'ECO Term Name', 'Unnamed: 13'],
      dtype='object')


,ZFIN ID,ZFIN Symbol,ZFIN Name,Human Symbol,Human Name,OMIM ID,Gene ID,HGNC ID,Evidence,Pub ID,ZFIN Abbreviation Name,ECO ID,ECO Term Name,Unnamed: 13
0,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-060313-16,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,
1,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-070210-39,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,
2,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-071118-46,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,
3,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-150121-5,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,
4,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-150226-13,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,


In [21]:
import pandas as pd
import re

# Example:
# zfin_df = your loaded ZFIN dataframe
# human_df = your human gene/function dataframe

zfin_df.columns = (
    zfin_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Remove extra empty column if present
if "unnamed:_13" in zfin_df.columns:
    zfin_df = zfin_df.drop(columns=["unnamed:_13"])

zfin_df["zfin_symbol_clean"] = zfin_df["zfin_symbol"].str.strip().str.lower()
zfin_df["human_symbol_clean"] = zfin_df["human_symbol"].str.strip().str.upper()

In [22]:
zfin_df.head()
zfin_df.shape

(44010, 15)

In [23]:
zfin_df.head()

,zfin_id,zfin_symbol,zfin_name,human_symbol,human_name,omim_id,gene_id,hgnc_id,evidence,pub_id,zfin_abbreviation_name,eco_id,eco_term_name,zfin_symbol_clean,human_symbol_clean
0,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-060313-16,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,ppardb,PPARD
1,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-070210-39,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,ppardb,PPARD
2,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-071118-46,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,ppardb,PPARD
3,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-150121-5,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,ppardb,PPARD
4,ZDB-GENE-000112-47,ppardb,peroxisome proliferator-activated receptor del...,PPARD,peroxisome proliferator activated receptor delta,600409,5467,9235,AA,ZDB-PUB-150226-13,Amino acid sequence comparison,ECO:0000031,protein BLAST evidence used in manual assertion,ppardb,PPARD


In [24]:
import re

# 1. Extract all gene symbols from df["Gene Names"]
human_gene_symbols = set()

for gene_names in df["Gene Names"].dropna():
    symbols = re.split(r"[\s,;]+", str(gene_names).strip().upper())
    human_gene_symbols.update(symbols)

# 2. Clean ZFIN symbols
zfin_df["zfin_symbol_clean"] = zfin_df["zfin_symbol"].astype(str).str.strip().str.upper()

# 3. Match ZFIN Symbol with Gene Names
matched_zfin_df = zfin_df[
    zfin_df["zfin_symbol_clean"].isin(human_gene_symbols)
].copy()

# 4. Count unique matched ZFIN genes
total_matched_genes = matched_zfin_df["zfin_symbol"].nunique()

print("Total matched ZFIN genes:", total_matched_genes)

Total matched ZFIN genes: 9185


In [25]:
matched_zfin_df[["zfin_id", "zfin_symbol", "zfin_name"]].drop_duplicates().head()

,zfin_id,zfin_symbol,zfin_name
41,ZDB-GENE-000208-18,urod,uroporphyrinogen decarboxylase
43,ZDB-GENE-000208-20,mixl1,Mix paired-like homeobox
45,ZDB-GENE-000208-21,kifc1,kinesin family member C1
51,ZDB-GENE-000208-23,col11a2,"collagen, type XI, alpha 2"
60,ZDB-GENE-000208-4,zic1,"zic family member 1 (odd-paired homolog, Droso..."


In [64]:
matched_zfin_df.shape

(22981, 15)

In [65]:
matched_zfin_df[["zfin_id", "zfin_symbol", "zfin_name"]].drop_duplicates().to_csv(
    "matched_zfin_genes.csv",
    index=False
)